# DroneFuse - Step 3: fusion with missing sensors and unknown-drone rejection (GPU)

Trains and evaluates, on the Step 2 features, for all 11 protocols and 3 seeds:
- **single-sensor** models (camera, audio, Avia, Mid-360, radar) and **late fusion** of them;
- **feature concatenation**, an **LMMUAVC-style** missing-slot model, and **DroneFuse** (attention that skips
  missing sensors, trained by removing sensors in the co-missing patterns MMAUD really shows), plus two
  DroneFuse variants (i.i.d. dropping; mixed dropping);
- unknown-type rejection with max-softmax, energy and **availability-normalised energy**;
- test-time sensor removal: every subset of sensor groups, and extra i.i.d. / pattern removal.

Everything runs twice when the Step 2b crops are attached: once with the whole-frame camera embeddings
(`runs/`, `tables/`) and once with the drone-centred crops (`runs_crop/`, `tables_crop/`). The two
cross-recording protocols (`xrec_*`) train on one flight of each car-park type and test on the other.

Everything that is chosen (presence threshold, best epoch, rejection thresholds) is chosen on **validation**;
validation never contains an unknown type. The test splits are only scored.

**Kaggle settings:** Accelerator = **GPU T4** (one GPU is enough). Internet = Off is fine (nothing is downloaded).
**Inputs to add:** `dronefuse-code` (latest version), `dronefuse-step2-features` and `dronefuse-step2b-crops`
(without the crops only the whole-frame variant runs).
To continue an interrupted run, also add the output of that earlier Step 3 run: finished runs are skipped.
**Safe for "Save & Run All":** one step per protocol; an error is recorded and the next protocol runs;
`step3_report.txt` is always written. **Expected run time:** a few GPU-hours in total; the first run took 3.7 h because it ran without a GPU.
If no GPU is found the notebook stops at the start with a message saying how to switch it on.

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
from dronefuse.fusion import TrainSettings
from dronefuse.windows import PROTOCOLS

INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/dronefuse_outputs/step3')
SEEDS = [0, 1, 2]
RUN_PROTOCOLS = list(PROTOCOLS)     # closed, cross-site, leave-one-type-out, cross-recording
CAMERA_VARIANTS = ['full', 'crop']  # 'crop' runs only when the Step 2b crops are attached
TIME_BUDGET_MINUTES = 11 * 60
ALLOW_CPU = False                 # True only to run without a GPU on purpose (much slower)
SETTINGS = TrainSettings(epochs=40, batch_size=256, lr=1e-3, weight_decay=1e-2)

In [ ]:
import json

from dronefuse import environment
from dronefuse.fusion import load_step2
from dronefuse.runner import StepRunner
from dronefuse.step3 import availability_table, device_name, resume_from
from dronefuse.windows import load_windows

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}


def save_summary():
    summary['steps'] = runner.steps
    with open(OUTPUT / 'step3_summary.json', 'w') as f:
        json.dump(summary, f, indent=1, default=str)


summary['environment'] = runner.run('measure the machine', environment.collect)
summary['gpu'] = environment.check_gpu(ALLOW_CPU)   # stops here with a clear message if there is no GPU
windows_csv = code_root / 'splits' / 'windows.csv'
windows = load_windows(windows_csv)

# The Step 2 features: the folder that holds the <recording>.deep.npz files.
deep_files = sorted(p for p in INPUT.rglob('*.deep.npz') if 'runs' not in p.parts)
if not deep_files:
    raise RuntimeError('No Step 2 features found. Add Input -> Your Datasets -> dronefuse-step2-features, '
                       'then Save & Run All again.')
STEP2 = deep_files[0].parent
print('Step 2 features:', STEP2, f'({len(deep_files)} recordings)')

# The Step 2b crops, if attached: the folder that holds the <recording>.crop.npz files.
crop_files = sorted(INPUT.rglob('*.crop.npz'))
CROPS = crop_files[0].parent if crop_files else None
print('Step 2b crops:', CROPS or 'not attached - only the whole-frame variant runs')
variants = [v for v in CAMERA_VARIANTS if v == 'full' or CROPS is not None]

# An earlier Step 3 output attached as input: copy its finished runs, so they are not repeated.
for previous in sorted({p.parent.parent for p in INPUT.rglob('runs*/*_threshold.json')}):
    resume_from(previous, OUTPUT)

datasets = {}
for camera in variants:
    data = runner.run(f'load the features ({camera} camera)', load_step2, STEP2, windows, camera, CROPS)
    if data is None:
        continue
    datasets[camera] = data
    print(f'{camera} camera: {len(data)} covered windows; skipped:', data.skipped or 'none')
    summary[f'windows_{camera}'] = len(data)
    summary[f'skipped_recordings_{camera}'] = data.skipped
if datasets:
    summary['availability_any_point'] = availability_table(next(iter(datasets.values())), 1)
    print('sensor group present (fraction of windows, any point):', json.dumps(summary['availability_any_point']))
if 'crop' in datasets:
    summary['availability_crop'] = availability_table(datasets['crop'], 1)
    print('with crops (camera = drone seen):', json.dumps(summary['availability_crop']))
print('device:', device_name())
save_summary()

In [ ]:
# Shortcut check on the crops, train/val only (CPU, a minute): do crops of windows where the drone is seen still
# tell flight 1 from flight 2 of the same type (chance 0.5)? And how well do they tell the drone type?
from dronefuse.step2 import linear_probe, recording_probe

if CROPS is not None and 'crop' in datasets:
    names = sorted(set(datasets['crop'].recording))
    summary['crop_probe'] = runner.run('linear probe on the crops', linear_probe, STEP2, windows, names, print, CROPS)
    summary['crop_recording_probe'] = runner.run('recording probe on the crops', recording_probe, STEP2, windows,
                                                 names, CROPS)
    save_summary()

In [ ]:
# The experiments: one step per protocol (a few minutes each on a GPU).
from dronefuse.step3 import run_protocol

summary['protocols'] = {}
for camera, data in datasets.items():
    for protocol in RUN_PROTOCOLS:
        summary['protocols'][f'{camera}/{protocol}'] = runner.run(
            f'protocol {protocol} ({camera} camera)', run_protocol, data, windows, protocol, SEEDS, OUTPUT, SETTINGS,
            camera=camera)
        save_summary()

In [ ]:
# Tables, figures and the report (numbers come only from the runs saved in OUTPUT/runs).
from dronefuse.step3 import format_step3_report, summarize, write_figures, write_tables

TITLE = {'full': 'CAMERA = WHOLE-FRAME EMBEDDINGS', 'crop': 'CAMERA = DRONE CROPS, PRESENT ONLY WHERE THE DRONE IS SEEN'}
sections = []
for camera, data in datasets.items():
    suffix = '' if camera == 'full' else f'_{camera}'
    results = runner.run(f'summarise the runs ({camera} camera)', summarize, data, windows, OUTPUT, RUN_PROTOCOLS,
                         tuple(SEEDS), camera)
    if results is None:
        continue
    with open(OUTPUT / f'results{suffix}.json', 'w') as f:
        json.dump(results, f, indent=1)
    summary[f'tables{suffix}'] = runner.run(f'LaTeX tables ({camera})', write_tables, results, OUTPUT / f'tables{suffix}')
    summary[f'figures{suffix}'] = runner.run(f'figures ({camera})', write_figures, results, OUTPUT / f'figures{suffix}')
    sections.append('=' * 30 + f' {TITLE[camera]} ' + '=' * 30 + '\n'
                    + format_step3_report(results, data, summary.get('environment')))
report = '\n\n'.join(sections) if sections else 'NO RESULTS - see the errors above.'
report += '\n\nAVAILABILITY (any point): ' + json.dumps(summary.get('availability_any_point'))
if summary.get('availability_crop'):
    report += '\nAVAILABILITY with crops (camera = drone seen): ' + json.dumps(summary['availability_crop'])
probe_lines = []
for block, r in (summary.get('crop_probe') or {}).items():
    if r and 'val_accuracy' in r:
        probe_lines.append(f"  drone type  {block:26s} val balanced acc {r['val_balanced_accuracy']:.3f} "
                           f"({r['train_windows']} train / {r['val_windows']} val windows)")
for block, r in (summary.get('crop_recording_probe') or {}).items():
    if r:
        probe_lines.append(f"  recording   {block:26s} balanced acc {r['balanced_accuracy']:.3f}  per type "
                           f"(M300, M30, P4) {r['per_type']}  ({r.get('windows', '?')} windows)")
if probe_lines:
    report += '\n\nSHORTCUT CHECK (train -> val; recording chance 0.500)\n' + '\n'.join(probe_lines)
failed = runner.failed()
report += '\n\nSTEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed))
(OUTPUT / 'step3_report.txt').write_text(report)
save_summary()
print(report)

## After the run

1. Send back `dronefuse_outputs/step3/step3_report.txt` (or the printed report above).
2. Open this notebook's saved version -> **Output** -> **New Dataset**, name it **`dronefuse-step3-results`**
   (private). It holds every run (logits included), so new analyses need no retraining.